# L3: On-Device Embeddings with AI Hub

In this lesson, you'll compile an embedding model for a Snapdragon device using **AI Hub**, run inference on real hardware, and store the embeddings in **Qdrant Edge**.

You'll learn to:
- Load a pre-trained embedding model from `qai_hub_models`
- Trace and compile it for a Snapdragon device via AI Hub
- Profile on-device latency and memory usage
- Run inference on a real Snapdragon device in the cloud
- Store the resulting embeddings in an EdgeShard
- Search across text and image memories

## Setup

In [ ]:
!pip install qdrant-edge-py qai-hub "qai-hub-models[nomic_embed_text,openai_clip]" torch transformers Pillow numpy

## 1. Configure AI Hub

AI Hub compiles and runs models on real Snapdragon hardware in the cloud. You submit a model, choose a target device, and AI Hub returns a compiled binary optimized for that chipset.

You'll need an API token from [AI Hub](https://aihub.qualcomm.com/).

In [ ]:
import os
import sys
sys.path.append("..")

import qai_hub
from utils import get_ai_hub_api_token, get_random_device

ai_hub_api_token = get_ai_hub_api_token()
!qai-hub configure --api_token $ai_hub_api_token

# Select a random Snapdragon device from the AI Hub fleet
device_name = get_random_device()
device = qai_hub.Device(device_name)
print(f"Target device: {device_name}")

## 2. Load a Text Embedding Model

We'll use `nomic_embed_text` from `qai_hub_models`. This is a transformer-based model that produces 512-dimensional text embeddings, optimized for on-device deployment.

The `qai_hub_models` package provides pre-trained models ready for compilation to Snapdragon hardware.

In [ ]:
import torch
from qai_hub_models.models.nomic_embed_text import Model as NomicEmbedText

text_model = NomicEmbedText.from_pretrained()

# Check the model's expected input specification
input_spec = text_model.get_input_spec()
print("Model input spec:")
for name, shape in input_spec.items():
    print(f"  {name}: {shape}")

EMBEDDING_DIM = 512
print(f"\nOutput: [{1}, {EMBEDDING_DIM}] embedding vector")

## 3. Set Up the Tokenizer

The embedding model expects tokenized input. We use the model's tokenizer to convert text to token IDs and attention masks.

Nomic uses prefixes to distinguish query vs. document embeddings:
- `search_query: ` for queries
- `search_document: ` for documents being stored

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5")
MAX_SEQ_LEN = 512

def tokenize(texts, prefix="search_document: "):
    """Tokenize texts for the nomic embedding model."""
    prefixed = [prefix + t for t in texts]
    encoded = tokenizer(
        prefixed,
        padding="max_length",
        truncation=True,
        max_length=MAX_SEQ_LEN,
        return_tensors="pt",
    )
    return encoded["input_ids"], encoded["attention_mask"]

# Test tokenization
sample_ids, sample_mask = tokenize(["hello world"])
print(f"Token IDs shape: {sample_ids.shape}")
print(f"Attention mask shape: {sample_mask.shape}")

## 4. Local Inference (Baseline)

Before compiling for a device, run the model locally with PyTorch. This gives us a baseline embedding to compare against the on-device output.

In [ ]:
sample_text = "Team meeting about the new edge AI deployment strategy"

input_ids, attention_mask = tokenize([sample_text])

with torch.no_grad():
    local_embedding = text_model(input_ids, attention_mask)

print(f"Local embedding shape: {local_embedding.shape}")
print(f"First 10 values: {local_embedding[0, :10].tolist()}")
print(f"L2 norm: {torch.norm(local_embedding[0]).item():.4f}")

## 5. Trace the Model for Compilation

AI Hub requires a traced (TorchScript) model. Tracing captures the model's computation graph by running it with example inputs.

In [ ]:
# Create example inputs matching the model's input spec
example_ids = torch.randint(0, tokenizer.vocab_size, (1, MAX_SEQ_LEN))
example_mask = torch.ones(1, MAX_SEQ_LEN, dtype=torch.long)

traced_model = torch.jit.trace(text_model, (example_ids, example_mask))
print("Model traced successfully")

## 6. Compile for Snapdragon via AI Hub

Submit the traced model to AI Hub for compilation. The compiler optimizes the model for the target device's NPU, GPU, and CPU.

In [ ]:
compile_job = qai_hub.submit_compile_job(
    model=traced_model,
    input_specs={
        "input_ids": (1, MAX_SEQ_LEN),
        "attention_mask": (1, MAX_SEQ_LEN),
    },
    device=device,
)

print(f"Compile job submitted: {compile_job.job_id}")
print(f"Target device: {device_name}")
print(f"Status URL: {compile_job.url}")

In [ ]:
# Wait for compilation and get the target model
target_model = compile_job.get_target_model()
print(f"Compilation complete!")
print(f"Target model: {target_model}")

## 7. Profile On-Device Performance

Profile the compiled model on the actual Snapdragon hardware. This shows you real-world latency, memory usage, and compute unit utilization.

In [ ]:
profile_job = qai_hub.submit_profile_job(
    model=target_model,
    device=device,
)

print(f"Profile job submitted: {profile_job.job_id}")

profile = profile_job.download_profile()
print(f"\nOn-device performance ({device_name}):")
print(f"  Inference time: {profile['execution_summary']['estimated_inference_time']}")
print(f"  Peak memory: {profile['execution_summary']['inference_memory_peak_range']}")

## 8. Run On-Device Inference

Now run the actual embedding model on the Snapdragon device. AI Hub sends the input to a real device and returns the output. This is the same result you'd get running the model locally on a phone.

In [ ]:
import numpy as np

# Prepare inputs for on-device inference
input_ids_np = input_ids.numpy().astype(np.int32)
attention_mask_np = attention_mask.numpy().astype(np.int32)

inference_job = qai_hub.submit_inference_job(
    model=target_model,
    inputs={"input_ids": [input_ids_np], "attention_mask": [attention_mask_np]},
    device=device,
)

print(f"Inference job submitted: {inference_job.job_id}")

# Download the on-device output
ondevice_output = inference_job.download_output_data()
ondevice_embedding = list(ondevice_output.values())[0][0]

print(f"\nOn-device embedding shape: {ondevice_embedding.shape}")
print(f"First 10 values: {ondevice_embedding[0, :10].tolist()}")

In [ ]:
# Compare local vs on-device embeddings
local_np = local_embedding.numpy().flatten()
device_np = ondevice_embedding.flatten()

cosine_sim = np.dot(local_np, device_np) / (np.linalg.norm(local_np) * np.linalg.norm(device_np))
print(f"Cosine similarity (local vs on-device): {cosine_sim:.6f}")
print(f"Max absolute difference: {np.max(np.abs(local_np - device_np)):.6f}")
print("\nThe on-device model produces nearly identical embeddings.")

## 9. Generate Embeddings and Store in Qdrant Edge

Now let's build a memory store. We'll embed a day's worth of observations and store them in Qdrant Edge.

For bulk embedding generation during development, we use local PyTorch inference. On a real device, the compiled model would run natively on the Snapdragon NPU.

In [ ]:
from pathlib import Path
from qdrant_edge import (
    EdgeShard, EdgeConfig, VectorDataConfig, Distance,
    Point, UpdateOperation, Query, QueryRequest,
)
import time

SHARD_DIR = "./text_memory_shard"
Path(SHARD_DIR).mkdir(parents=True, exist_ok=True)

VECTOR_NAME = "text_embedding"

config = EdgeConfig(
    vector_data={
        VECTOR_NAME: VectorDataConfig(
            size=EMBEDDING_DIM,
            distance=Distance.Cosine,
        )
    }
)

text_shard = EdgeShard(SHARD_DIR, config)
print("EdgeShard ready for text memories")

In [ ]:
memories = [
    {"text": "Team standup meeting discussed the new API design", "location": "office"},
    {"text": "Grabbed coffee at the cafe on 5th street", "location": "cafe"},
    {"text": "Reviewed pull request for the authentication module", "location": "office"},
    {"text": "Lunch at the Thai restaurant, ordered pad see ew", "location": "restaurant"},
    {"text": "Whiteboard session about database migration strategy", "location": "office"},
    {"text": "Picked up groceries: milk, eggs, bread, avocados", "location": "store"},
    {"text": "Evening run along the waterfront trail, 5k in 24 minutes", "location": "outdoors"},
    {"text": "Read a paper about transformer attention mechanisms", "location": "home"},
    {"text": "Video call with the ML team about model quantization", "location": "home"},
    {"text": "Debugged a memory leak in the edge inference pipeline", "location": "office"},
]

# Batch-embed all memories using local inference
texts = [m["text"] for m in memories]
input_ids_batch, mask_batch = tokenize(texts)

with torch.no_grad():
    all_embeddings = text_model(input_ids_batch, mask_batch)

# Store in EdgeShard
base_time = time.time() - len(memories) * 3600

points = []
for i, (memory, emb) in enumerate(zip(memories, all_embeddings)):
    points.append(Point(
        id=i,
        vector={VECTOR_NAME: emb.tolist()},
        payload={
            "text": memory["text"],
            "location": memory["location"],
            "timestamp": base_time + i * 3600,
        }
    ))

text_shard.update(UpdateOperation.upsert_points(points))
print(f"Stored {len(points)} text memories in Qdrant Edge")

## 10. Semantic Search Over Memories

Query your memory store using natural language. The query is embedded with `search_query:` prefix and compared against stored documents.

In [ ]:
def search_memory(query_text, limit=3):
    """Embed a query and search the text memory shard."""
    ids, mask = tokenize([query_text], prefix="search_query: ")
    with torch.no_grad():
        query_emb = text_model(ids, mask)
    
    results = text_shard.query(
        QueryRequest(
            query=Query.Nearest(query_emb[0].tolist(), using=VECTOR_NAME),
            limit=limit,
            with_vector=False,
            with_payload=True,
        )
    )
    return results

queries = [
    "What did I eat today?",
    "Any work meetings about technical design?",
    "What exercise did I do?",
]

for q in queries:
    print(f"\nQuery: {q}")
    results = search_memory(q)
    for r in results:
        print(f"  [{r.score:.3f}] {r.payload['text']}")

## 11. Image Embeddings with CLIP

Edge devices with cameras (smart glasses, robots) need visual memory. CLIP from `qai_hub_models` produces 512-dimensional image embeddings that live in the same space as text embeddings, enabling cross-modal search.

In [ ]:
from qai_hub_models.models.openai_clip import Model as OpenAIClip
from PIL import Image

clip_model = OpenAIClip.from_pretrained()
print("CLIP model loaded")

# Create sample images (colored rectangles simulating scenes)
IMAGE_DIR = "./sample_images"
Path(IMAGE_DIR).mkdir(parents=True, exist_ok=True)

labels = ["red object", "green landscape", "blue sky", "yellow sign", "purple flower"]
colors = [(255, 0, 0), (0, 180, 0), (0, 100, 255), (255, 255, 0), (180, 0, 255)]

image_paths = []
for i, (label, color) in enumerate(zip(labels, colors)):
    img = Image.new("RGB", (224, 224), color)
    path = f"{IMAGE_DIR}/image_{i}.png"
    img.save(path)
    image_paths.append(path)

print(f"Created {len(image_paths)} sample images")

In [ ]:
# Generate image embeddings using CLIP's visual encoder
VISION_SHARD_DIR = "./vision_memory_shard"
Path(VISION_SHARD_DIR).mkdir(parents=True, exist_ok=True)

VISION_VECTOR_NAME = "image_embedding"

vision_config = EdgeConfig(
    vector_data={
        VISION_VECTOR_NAME: VectorDataConfig(
            size=EMBEDDING_DIM,
            distance=Distance.Cosine,
        )
    }
)

vision_shard = EdgeShard(VISION_SHARD_DIR, vision_config)

# Embed images using CLIP's visual encoder
vision_points = []
for i, (path, label) in enumerate(zip(image_paths, labels)):
    img = Image.open(path)
    img_tensor = clip_model.image_preprocessor(img).unsqueeze(0)

    with torch.no_grad():
        features = clip_model.clip.encode_image(img_tensor)
        features = features / features.norm(dim=1, keepdim=True)
        embedding = features.squeeze(0).float().tolist()

    vision_points.append(Point(
        id=i,
        vector={VISION_VECTOR_NAME: embedding},
        payload={
            "image_path": path,
            "label": label,
            "timestamp": time.time() - (len(labels) - i) * 600,
        }
    ))

vision_shard.update(UpdateOperation.upsert_points(vision_points))
print(f"Stored {len(vision_points)} image memories")

In [ ]:
# Cross-modal search: query images using text
import clip

text_query = "something red"
text_tokens = clip.tokenize([text_query])

with torch.no_grad():
    text_features = clip_model.clip.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=1, keepdim=True)
    query_emb = text_features.squeeze(0).float().tolist()

results = vision_shard.query(
    QueryRequest(
        query=Query.Nearest(query_emb, using=VISION_VECTOR_NAME),
        limit=3,
        with_vector=False,
        with_payload=True,
    )
)

print(f"Text query: '{text_query}'")
print("Matching images:")
for r in results:
    print(f"  [{r.score:.3f}] {r.payload['label']}")

## 12. Cleanup

In [ ]:
text_shard.close()
vision_shard.close()

import shutil
shutil.rmtree(SHARD_DIR, ignore_errors=True)
shutil.rmtree(VISION_SHARD_DIR, ignore_errors=True)
shutil.rmtree(IMAGE_DIR, ignore_errors=True)
print("Cleaned up")

## Summary

In this lesson you learned how to:
- Load embedding models from `qai_hub_models` (nomic_embed_text, CLIP)
- Trace a model with `torch.jit.trace()` for compilation
- Compile for a Snapdragon device via `qai_hub.submit_compile_job()`
- Profile on-device performance with `qai_hub.submit_profile_job()`
- Run inference on real hardware with `qai_hub.submit_inference_job()`
- Store embeddings in Qdrant Edge and perform semantic search
- Use CLIP for cross-modal text-to-image search

In the next lesson, you'll add contextual filtering to narrow your search results by time, location, and category.